In [2]:
import os
os.chdir("../")
os.getcwd()

'/scratch/pawsey1172/sasunih/DeepSpot'

In [1]:
from deepspot.utils.utils_image import get_morphology_model_and_preprocess
from deepspot.utils.utils_image import compute_mini_tiles
from deepspot.utils.utils_image import crop_tile

from pathlib import Path
from tqdm import tqdm
import scanpy as sc
import pandas as pd
import numpy as np
import pyvips
import torch
import glob
import yaml
import json
import anndata as ad

/scratch/pawsey1172/sasunih/miniconda3/envs/deepspot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [4]:
n_mini_tiles = 9 # number of non-overlaping sub-spots per subspot
image_feature_model = "inception" # foundation model 
samples = ["KC1", "KC2", "KC3", "LC1", "LC2", "LC3", "LC4", "LC5"] # Tertiary Lymphoid Kidney and Lung dataset
out_folder = "TLS_VISIUM_USZ"

In [5]:
# load pathology foundation model and preprocessing pipeline
morphology_model, preprocess, feature_dim = get_morphology_model_and_preprocess(model_name=image_feature_model, 
                                                                                device=device)
print(feature_dim)


2048


In [9]:
# Gene Preprocess Across all samples, as according to paper
slices_dict = {}
for sample in samples:
    adata_in = f"TLS_VISIUM_USZ/h5ad_preprocessed/{sample}.h5ad"
    adata = sc.read_h5ad(adata_in)
    slices_dict[sample] = adata

# Combine into a single master AnnData object
# index_unique=None keeps the original barcodes so your index renaming logic works
adata_bulk = ad.concat(slices_dict, label="tissue_slice", index_unique=None)

# --- 2. Prepare and Run the Paper's Pipeline ---
# Seurat_v3 requires raw counts. We save a copy to 'raw_counts' layer first.
adata_bulk.layers["raw_counts"] = adata_bulk.X.copy()

# Normalize total counts per spot to 10,000 and log1p transform
sc.pp.normalize_total(adata_bulk, target_sum=1e4)
sc.pp.log1p(adata_bulk)

# Calculate top 5000 highly variable genes globally using the batch key
sc.pp.highly_variable_genes(
    adata_bulk, 
    flavor='seurat_v3', 
    n_top_genes=5000,
    batch_key="tissue_slice",
    layer="raw_counts"
)
print('Calculated top 5000 highly variable genes across all samples')

# --- 3. Save the Global CSV File (Across All Samples) ---
adata_bulk.var["isPredicted"] = adata_bulk.var.highly_variable.values
adata_bulk.var["gene_name"] = adata_bulk.var.index.values

print(adata_bulk.var)

Path(f"{out_folder}/data").mkdir(parents=True, exist_ok=True)
adata_bulk.var.to_csv(f"{out_folder}/data/info_highly_variable_genes_Visium.csv", index=False)

# --- 4. Save individual Count Matrices to Pickle (Per Sample) ---
print('Saving counts to pickle files per sample...')

# Loop through each unique sample name present in the batch key
for sample in adata_bulk.obs["tissue_slice"].unique():
    # Subset the master adata to only include this specific sample's rows
    adata_sample = adata_bulk[adata_bulk.obs["tissue_slice"] == sample].copy()
    
    # Convert its processed expression matrix (X) to a DataFrame
    # If adata.X is a sparse matrix, .toarray() ensures it converts cleanly to pandas
    X_matrix = adata_sample.X.toarray() if hasattr(adata_sample.X, "toarray") else adata_sample.X
    
    counts = pd.DataFrame(
        X_matrix, 
        index=adata_sample.obs_names.values, 
        columns=adata_sample.var.index
    )
    
    # Apply your specific barcode naming convention: {barcode}_{sample}
    counts.index = [f"{b}_{sample}" for b in counts.index]
    
    print(f"Saving counts for {sample}: Shape {counts.shape}")
    counts.to_pickle(f"{out_folder}/data/inputX/{sample}.pkl")

/scratch/pawsey1172/sasunih/miniconda3/envs/deepspot/lib/python3.12/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Calculated top 5000 highly variable genes across all samples
         highly_variable  highly_variable_rank     means  variances  \
SAMD11              True                1987.0  0.038646   0.041361   
NOC2L              False                3920.0  0.237111   0.658026   
KLHL17             False                4777.0  0.048214   0.057358   
PLEKHN1            False                4318.0  0.047431   0.062506   
PERM1              False                4374.0  0.006063   0.006109   
...                  ...                   ...       ...        ...   
TSPY1              False                   NaN  0.000000   0.000000   
USP9Y              False                   NaN  0.020168   0.025537   
DDX3Y              False                3812.5  0.284542   1.476013   
TMSB4Y             False                   NaN  0.007011   0.007705   
KDM5D              False                   NaN  0.253526   0.758448   

         variances_norm  highly_variable_nbatches  isPredicted gene_name  
SAMD11     

OSError: Cannot save file into a non-existent directory: 'TLS_VISIUM_USZ/data/inputX'

In [9]:
def sample_preprocess(sample, adata_in, json_path, img_path, out_folder, morphology_model, preprocess, feature_dim):
    '''Processing per image. 
    sample : str 
        sample name
    adata_in : str
        path to anndata object.
    json_path : str
        path to json object (spot diameter)
    img_path : str
        path to histology object
    out_folder : str
        path to output folder
    morphology_model : torch.model
        foundation model used for image embedding
    preprocess : torchvisions.transforms.Compose
        preprocessing transformations
    feature_dim : int
        Number of model features
    '''
    # create folder to save tile embeddings
    folder_to_create = f"{out_folder}/data/image_features/{image_feature_model}/{sample}"
    Path(folder_to_create).mkdir(parents=True, exist_ok=True)
    
    spot_diameter_fullres = round(json.load(open(json_path))["spot_diameter_fullres"]) # spot diameter
    print(spot_diameter_fullres)

    adata = sc.read_h5ad(adata_in)
    print('Loaded AnnData Object')
    print(adata)

    # Extract tile embeddings/representations per spot
    image = pyvips.Image.new_from_file(img_path)
    morphology_model = morphology_model.to(device)
    barcode = adata.obs_names
    x_pixel = adata.obs.x_pixel
    y_pixel = adata.obs.y_pixel


    image = pyvips.Image.new_from_file(img_path)
    main_features = np.zeros([len(adata), feature_dim])
    
    for i, (b, x, y) in tqdm(enumerate(zip(barcode, x_pixel, y_pixel))): 

        main_tile = crop_tile(image, x, y, spot_diameter_fullres)
        preprocess_main_tile = preprocess(main_tile)

        X = np.zeros([n_mini_tiles + 1, 3, preprocess_main_tile.shape[1], preprocess_main_tile.shape[1]]) 
        X[0, :] = preprocess_main_tile

        mini_tiles = compute_mini_tiles(main_tile, n_mini_tiles)
    
        for j, mini_tile in enumerate(mini_tiles):
        
            X[j+1, :] = preprocess(mini_tile)

    
        X = torch.from_numpy(X)
        X = X.to(device).float()
        # We recommend using mixed precision for faster inference.
        with torch.autocast(device_type="cuda", dtype=torch.float32):
            with torch.inference_mode():
                output = morphology_model(X)
                output = output.float().detach().cpu().numpy()

        main_features[i,:] = output[0]
    
        np.save(f"{folder_to_create}/{b}.npy", output)

    return(glob.glob(f"{out_folder}/data/image_features/{image_feature_model}/{sample}/*")[:10])

In [10]:
for sample in samples:
    print(sample)
    # Input files
    adata_in = f"TLS_VISIUM_USZ/h5ad_preprocessed/{sample}.h5ad"
    json_path = f"TLS_VISIUM_USZ/h5ad_preprocessed/{sample}.json"
    img_path = f"TLS_VISIUM_USZ/tif_slides/{sample}.tif"

    out = sample_preprocess(sample, adata_in, json_path, img_path, out_folder, morphology_model, preprocess, feature_dim)

    print('Numpy matrices of patches')
    print(out)
    print()
    print()

82
AnnData object with n_obs × n_vars = 3690 × 17845
    obs: 'in_tissue', 'x_array', 'y_array', 'x_pixel', 'y_pixel', 'manual_anno', 'manual_anno_tls', 'aestetik_manual_anno', 'aestetik_manual_anno_tls', 'ground_truth', 'blur_score'
    var: 'gene_ids', 'feature_types', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial'
AnnData object with n_obs × n_vars = 3690 × 17845
    obs: 'in_tissue', 'x_array', 'y_array', 'x_pixel', 'y_pixel', 'manual_anno', 'manual_anno_tls', 'aestetik_manual_anno', 'aestetik_manual_anno_tls', 'ground_truth', 'blur_score'
    var: 'gene_ids', 'feature_types', 'gene_name', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'spatial', 'hvg'
    obsm: 'spatial'
                gene_ids    feature_types gene_name  highly_variable  \
SAMD11   ENSG00000187634  Gene Expression    SAMD11             True   
NOC2L    ENSG00000188976  Gene Expression     NOC2L            False   
KLHL17   ENSG00000187961  Gene Expression    KLHL17

3690it [02:00, 30.69it/s]


['TLS_VISIUM_USZ/data/image_features/inception/KC1/GCTGAGCAACGGTTCT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/GCTAGCACCTGGGCCA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/CGCCGTTCAGCATAGT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/TCAAAGAGCTATCTGT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/TATTAACACCAAAGCA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/ACCAGTGCCCGGTCAA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/GAAGCCACTGATTATG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/CCGCACGTGACCTCGG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/GAAACCGAATTACCTT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC1/GGCGCATGAATTGATG-1.npy']


83
AnnData object with n_obs × n_vars = 1870 × 17845
    obs: 'in_tissue', 'x_array', 'y_array', 'x_pixel', 'y_pixel', 'manual_anno', 'manual_anno_tls', 'aestetik_manual_anno', 'aestetik_manual_anno_tls', 'ground_truth', 'blur_score'
    var: 'gen

1870it [00:59, 31.21it/s]


['TLS_VISIUM_USZ/data/image_features/inception/KC2/GCTAGCACCTGGGCCA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/CGCCGTTCAGCATAGT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/GAACTGTGGAGAGACA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/GAAGCCACTGATTATG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/GAAACCGAATTACCTT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/GGCGCATGAATTGATG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/TCTGACGGGCTAACCC-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/CGACTCAGGATGTTAT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/GCAACAGCAGTATGCG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/KC2/TCACAGGGAATCGCAA-1.npy']


83
AnnData object with n_obs × n_vars = 2654 × 17845
    obs: 'in_tissue', 'x_array', 'y_array', 'x_pixel', 'y_pixel', 'manual_anno', 'manual_anno_tls', 'aestetik_manual_anno', 'aestetik_manual_anno_tls', 'ground_truth', 'blur_score'
    var: 'gen

...

['TLS_VISIUM_USZ/data/image_features/inception/LC3/GCTGAGCAACGGTTCT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/GCTAGCACCTGGGCCA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/CGCCGTTCAGCATAGT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/TCAAAGAGCTATCTGT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/TATTAACACCAAAGCA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/ACCAGTGCCCGGTCAA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/TAAGAGGGACAGGGAC-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/GAAGCCACTGATTATG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/CCGCACGTGACCTCGG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC3/GAAACCGAATTACCTT-1.npy']


82
AnnData object with n_obs × n_vars = 2939 × 17845
    obs: 'in_tissue', 'x_array', 'y_array', 'x_pixel', 'y_pixel', 'manual_anno', 'manual_anno_tls', 'aestetik_manual_anno', 'aestetik_manual_anno_tls', 'ground_truth', 'blur_score'
    var: 'gen

2939it [01:39, 29.41it/s]


['TLS_VISIUM_USZ/data/image_features/inception/LC4/GCTGAGCAACGGTTCT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/TCAAAGAGCTATCTGT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/TATTAACACCAAAGCA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/ACCAGTGCCCGGTCAA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/GGGAGTAATGGCTGGC-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/CCGCACGTGACCTCGG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/GAAACCGAATTACCTT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/GGCGCATGAATTGATG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/TACGGGATGCTAGCAG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC4/GTGTGAATAACTTAGG-1.npy']


83
AnnData object with n_obs × n_vars = 2017 × 17845
    obs: 'in_tissue', 'x_array', 'y_array', 'x_pixel', 'y_pixel', 'manual_anno', 'manual_anno_tls', 'aestetik_manual_anno', 'aestetik_manual_anno_tls', 'ground_truth', 'blur_score'
    var: 'gen

2017it [01:05, 30.70it/s]

['TLS_VISIUM_USZ/data/image_features/inception/LC5/GCTAGCACCTGGGCCA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/CGCCGTTCAGCATAGT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/TCAAAGAGCTATCTGT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/TATTAACACCAAAGCA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/GAAACCGAATTACCTT-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/GGCGCATGAATTGATG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/GCAAGCTGGAAACCGC-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/GCAACAGCAGTATGCG-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/TCACAGGGAATCGCAA-1.npy', 'TLS_VISIUM_USZ/data/image_features/inception/LC5/CCACTGTTTGGATTAA-1.npy']


